# 第七节 LeNet-5-首个商用级别卷积神经网络

## 实验目标
通过本案例的学习：

1. 了解LeNet-5的网络结构；


## 注意事项

1. 本案例推荐使用Pytorch-1.0.0，需使用 <font color='red' >GPU</font> 运行，请查看[《ModelArts CodeLab介绍》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0010.html#section3)了解切换硬件规格的方法；

2. 如果您是第一次使用 JupyterLab，请查看[《ModelArts JupyterLab使用指导》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0012.html)了解使用方法；

3. 如果您在使用 JupyterLab 过程中碰到报错，请参考[《ModelArts JupyterLab常见问题解决办法》](https://support.huaweicloud.com/modelarts_faq/modelarts_05_0185.html)尝试解决问题。

## 实验步骤

## 案例内容介绍
上一节我们使用一个三层的CNN网络，在训练3000个epoch之后，达到了0.9673的准确率。其实MNIST数据集之所以成为深度学习入门的数据集，是因为LeNet-5网络的诞生，该网络的手写数字识别效果能达到商用水平，是第一个真正意义上达到商用的深度神经网络，广泛应用于手写支票的识别。  
本案例将使用LeNet-5来实现手写数字识别。

### 1. 加载数据集
直接复用上一节的代码

In [1]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '../datasets/MNIST_data'))
from load_data_all import load_data_all

datasets_dir = '../datasets'
train_x, train_y, test_x, test_y = load_data_all(datasets_dir)
train_x = train_x.view(-1, 1, 28, 28)
test_x = test_x.view(-1, 1, 28, 28)

INFO:root:Using MoXing-v1.17.3-

INFO:root:Using OBS-Python-SDK-3.20.7


训练集规模： 60000 ，测试集规模： 10000


### 2. 构建LeNet-5网络和评价函数
LeNet-5有5层网络，分别是卷积层1、卷积层2、全连接层1、全连接层2、全连接层3，网络结构如下图所示：
[LeNet-5.jpg](https://modelarts-labs-bj4.obs.cn-north-4.myhuaweicloud.com:443/course/hwc_edu/deep_learning/datasets/imgs/LeNet-5.jpg)
由于网络层数变多，本次网络结构定义的代码中，我们将使用Pytorch中的nn.Sequential来简化网络的定义过程。nn.Sequential用于构建序列化模型时非常方便，可以直接把多层结构拼装成一个层，代码如下，具体含义请查看代码注释

In [2]:
import torch
from torch import nn

class Network(nn.Module):
    def __init__(self, num_of_weights):
        torch.manual_seed(0)
        super().__init__()
        self.conv1 = nn.Sequential( # input_size=(1*28*28)
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=2),  # padding=2保证输入输出尺寸相同，output_size=(6*28*28)
            nn.ReLU(),  # input_size=(6*28*28)
            nn.MaxPool2d(kernel_size=2, stride=2))  # output_size=(6*14*14)
        
        self.conv2 = nn.Sequential( # input_size=(6*14*14)
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0),  # output_size=(16*10*10)
            nn.ReLU(),  # input_size=(16*10*10)
            nn.MaxPool2d(kernel_size=2, stride=2))  # input_size=(16*5*5)
        
        self.fc1 = nn.Sequential(
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU()
        )
        
        self.fc2 = nn.Sequential(
            nn.Linear(120, 84),
            nn.ReLU()
        )
        
        self.fc3 = nn.Linear(84, 10)
    
    def forward(self, x):
        """
        前向传播函数
        """
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size()[0], -1)
        x = self.fc1(x)
        x = self.fc2(x)
        out = self.fc3(x)
        
        return out
        
    def evaluate(self, pred_y, true_y):
        """
        准确率统计函数，与上一节一致
        """
        pred_labels = torch.argmax(pred_y, dim=1)
        acc = (pred_labels == true_y).float().mean()
        return acc

### 3. 交叉熵损失函数
与上一节代码一致

In [3]:
import torch.nn.functional as F
loss_fun = F.cross_entropy

### 4. 实现GPU训练的梯度下降算法
与上一节代码一致

In [4]:
net = Network(28*28)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

### 5. 实现训练函数
与上一节代码一致

In [5]:
def train(net, train_x, train_y, test_x, test_y, max_epochs=100):
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    for epoch in range(1, max_epochs + 1):
        net.train()  # 切换为训练模式
        train_x, train_y = train_x.to(device), train_y.to(device)
        pred_y_train = net.forward(train_x)  # 前向传播
        train_loss = loss_fun(pred_y_train, train_y)  # 计算损失

        # 计算梯度，更新权值
        train_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (epoch == 1) or (epoch % 200 == 0):
            net.eval()  # 切换为评价模式，评价模式不计算梯度，计算更快
            test_x, test_y = test_x.to(device), test_y.to(device)
            pred_y_test = net.forward(test_x)
            test_loss = loss_fun(pred_y_test, test_y)
            train_acc = net.evaluate(pred_y_train, train_y)
            test_acc = net.evaluate(pred_y_test, test_y)
            print('epoch %d, train_loss %.4f, test_loss %.4f, train_acc: %.4f, test_acc: %.4f' % (epoch, train_loss.item(), test_loss.item(), train_acc, test_acc))
    return train_losses, test_losses, train_accs, test_accs

### 6. 开始训练
代码与上一节一致  
训练耗时约16分钟

In [6]:
import time
start_time = time.time()
max_epochs = 3000
train_losses, test_losses, train_accs, test_accs = train(net, train_x, train_y, test_x, test_y, max_epochs=max_epochs)
print('cost time: %.1f s' % int(time.time() - start_time))

epoch 1, train_loss 2.3047, test_loss 2.3046, train_acc: 0.0992, test_acc: 0.1009

epoch 200, train_loss 2.3004, test_loss 2.3003, train_acc: 0.0992, test_acc: 0.1010

epoch 400, train_loss 2.2950, test_loss 2.2948, train_acc: 0.1622, test_acc: 0.1659

epoch 600, train_loss 2.2869, test_loss 2.2866, train_acc: 0.2486, test_acc: 0.2488

epoch 800, train_loss 2.2684, test_loss 2.2676, train_acc: 0.2753, test_acc: 0.2657

epoch 1000, train_loss 2.1952, test_loss 2.1926, train_acc: 0.3383, test_acc: 0.3201

epoch 1200, train_loss 1.2882, test_loss 1.2614, train_acc: 0.7085, test_acc: 0.7114

epoch 1400, train_loss 0.5569, test_loss 0.5376, train_acc: 0.8310, test_acc: 0.8396

epoch 1600, train_loss 0.4516, test_loss 0.4337, train_acc: 0.8595, test_acc: 0.8667

epoch 1800, train_loss 0.3659, test_loss 0.3510, train_acc: 0.8902, test_acc: 0.8952

epoch 2000, train_loss 0.3174, test_loss 0.3034, train_acc: 0.9047, test_acc: 0.9083

epoch 2200, train_loss 0.2801, test_loss 0.2668, train_acc: 0

至此，本案例完成。